In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
cardsdf=pd.read_csv("/content/drive/MyDrive/FraudDetection/cards_data.csv")
cardsdf.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [ ]:
usersdf=pd.read_csv("/content/drive/MyDrive/FraudDetection/users_data.csv")
usersdf.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [ ]:
transactionsdf=pd.read_csv("/content/drive/MyDrive/FraudDetection/transactions_data.csv")
transactionsdf.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [ ]:
!pip install tpot scikit-learn

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 6.1 MB/s eta 0:00:00
  Created wheel for stopit: filename=stopit-1.1.2-py3-none-any.whl size=11939 sha256=32d61ccc06aa8077d2740ef293a90633b58d73a9f6a93357e68425852883a00d
  Stored in directory: /root/.cache/pip/wheels/da/77/2d/adbc56bc4db95ad80c6d4e71cd69e2d9d122174904342e3f7f
Successfully built stopit


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tpot import TPOTClassifier


In [ ]:
!pip install dask


In [ ]:
import dask.dataframe as dd

# Convert Pandas DataFrames to Dask DataFrames
transactions_ddf = dd.from_pandas(transactionsdf, npartitions=10)
cards_ddf = dd.from_pandas(cardsdf, npartitions=10)
users_ddf = dd.from_pandas(usersdf, npartitions=10)

# Perform Dask-based merge
df = transactions_ddf.merge(cards_ddf, left_on='card_id', right_on='id', how='left')

# Check if 'client_id' is present in df before the second merge
if 'client_id' in df.columns:
    df = df.merge(users_ddf, left_on='client_id', right_on='id', how='left')
else:
    print("Error: 'client_id' column not found in the DataFrame.")
    # Further investigation is needed to understand why 'client_id' is missing
    # You might need to check the 'transactions_ddf' or the first merge operation

# Convert back to Pandas
df = df.compute()

# Drop unnecessary columns
df.drop(columns=['id_card', 'id_user'], inplace=True, errors='ignore') # errors='ignore' to avoid error if columns don't exist

df.head()

/usr/local/lib/python3.11/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


Error: 'client_id' column not found in the DataFrame.


,id_x,date,client_id_x,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,9075438,2011-01-26 16:11:00,665,2798,$2.13,Online Transaction,39021,ONLINE,<NA>,NaN,...,Debit,5367612713630895,03/2022,457,YES,1,$1264,05/1998,2014,No
1,9075446,2011-01-26 16:12:00,1444,5965,$31.91,Online Transaction,18563,ONLINE,<NA>,NaN,...,Debit,5143782902956271,10/2023,803,YES,1,$22886,12/2009,2009,No
2,9075454,2011-01-26 16:14:00,246,1158,$28.11,Swipe Transaction,61541,Minneapolis,MN,55419.0,...,Debit,4957823817777302,03/2022,413,YES,1,$40350,02/2008,2014,No
3,9075460,2011-01-26 16:15:00,1302,5464,$56.71,Swipe Transaction,65698,Jefferson,NC,28640.0,...,Debit,5867658090755003,02/2024,205,YES,1,$9883,11/2006,2010,No
4,9075462,2011-01-26 16:15:00,1393,3018,$18.03,Swipe Transaction,96475,David City,NE,68632.0,...,Debit,4412200129683784,05/2021,930,YES,2,$8845,05/2009,2010,No


In [ ]:
import pandas as pd
import numpy as np

# Make a copy to avoid modifying the original DataFrame
df_cleaned = df.copy()

# Convert 'amount' column to numeric (remove '$' and convert to float)
df_cleaned['amount'] = df_cleaned['amount'].replace('[\$,]', '', regex=True).astype(float)

# Convert categorical variables into numerical (One-Hot Encoding)
df_cleaned = pd.get_dummies(df_cleaned, columns=['card_type', 'use_chip', 'card_on_dark_web'])

# Drop unnecessary columns (ID-related, card_number, CVV for security reasons)
df_cleaned.drop(columns=['id_x', 'card_id', 'merchant_id', 'card_number', 'cvv'], inplace=True, errors='ignore')

# Fill missing values with appropriate values for different data types
# Use an empty string for string columns and 0 for numeric columns
for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col].fillna('', inplace=True)  # Fill string columns with empty string
    elif df_cleaned[col].dtype in [np.number]:
        df_cleaned[col].fillna(0, inplace=True)    # Fill numeric columns with 0

df_cleaned.head()

In [ ]:
# Convert 'credit_limit' column to numeric (remove '$' and convert to float)
df_cleaned['credit_limit'] = df_cleaned['credit_limit'].replace('[\$,]', '', regex=True).astype(float)

# Convert 'acct_open_date' to year format
df_cleaned['acct_open_date'] = df_cleaned['acct_open_date'].astype(str).str[-4:].astype(int)

# Convert date column to datetime
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])

# Extract useful time features
df_cleaned['transaction_hour'] = df_cleaned['date'].dt.hour
df_cleaned['transaction_day'] = df_cleaned['date'].dt.day
df_cleaned['transaction_month'] = df_cleaned['date'].dt.month
df_cleaned['transaction_year'] = df_cleaned['date'].dt.year

# Drop original 'date' column
df_cleaned.drop(columns=['date'], inplace=True)


In [ ]:
df_cleaned.head()

In [ ]:
import pandas as pd

# Step 1: Drop existing 'is_fraud' column (if any)
df_cleaned = df_cleaned.drop(columns=["is_fraud"], errors="ignore")

# Step 2: Define fraud detection rules
df_cleaned["is_fraud"] = 0  # Default to non-fraud

# Condition 1: High transaction amount
df_cleaned.loc[df_cleaned["amount"] > 10000, "is_fraud"] = 1

# Condition 2: Multiple errors in transaction
df_cleaned.loc[df_cleaned["errors"].notna(), "is_fraud"] = 1

# Condition 3: Card found on dark web
df_cleaned.loc[df_cleaned["card_on_dark_web_No"] == False, "is_fraud"] = 1

# Condition 4: Unusual use_chip behavior (e.g., swipe for high-value)
df_cleaned.loc[(df_cleaned["amount"] > 5000) & (df_cleaned["use_chip_Swipe Transaction"] == True), "is_fraud"] = 1

# Condition 5: Transaction at an unusual merchant category (MCC)
fraud_mcc_codes = [4829, 7011, 5912]  # Example risky merchant categories
df_cleaned.loc[df_cleaned["mcc"].isin(fraud_mcc_codes), "is_fraud"] = 1

# Step 3: Check fraud distribution
print(df_cleaned["is_fraud"].value_counts())


In [ ]:
df_cleaned.head()

In [ ]:
print(df_cleaned["is_fraud"].value_counts())


In [ ]:
!pip install tpot scikit-learn imbalanced-learn


In [ ]:
import pandas as pd
import numpy as np
from tpot import TPOTClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix



In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Select Features (Exclude Non-Numeric & Target Column)
X = df_cleaned.drop(columns=["is_fraud", "merchant_city", "merchant_state", "card_brand", "errors"])
y = df_cleaned["is_fraud"]

# Step 2: Fill Missing Values (Replace NaN with 0 for now)
X = X.fillna(0)

# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}")



In [ ]:
print(X_train.dtypes)


In [ ]:
import pandas as pd

# Drop `id_y` since it's an identifier (not useful for training)
X_train.drop(columns=["id_y"], inplace=True, errors="ignore")
X_test.drop(columns=["id_y"], inplace=True, errors="ignore")

# Convert `expires` (MM/YYYY format) into separate columns
X_train["exp_month"] = X_train["expires"].str.split("/").str[0].astype(int)
X_train["exp_year"] = X_train["expires"].str.split("/").str[1].astype(int)
X_train.drop(columns=["expires"], inplace=True)

X_test["exp_month"] = X_test["expires"].str.split("/").str[0].astype(int)
X_test["exp_year"] = X_test["expires"].str.split("/").str[1].astype(int)
X_test.drop(columns=["expires"], inplace=True)

# Verify all features are numeric
print(X_train.dtypes)


In [ ]:
X_train["has_chip"] = X_train["has_chip"].map({"Yes": 1, "No": 0})
X_test["has_chip"] = X_test["has_chip"].map({"Yes": 1, "No": 0})

# Fill NaN values with a default (e.g., assume "No" if missing)
X_train["has_chip"] = X_train["has_chip"].fillna(0).astype(int)
X_test["has_chip"] = X_test["has_chip"].fillna(0).astype(int)


In [ ]:
import lightgbm as lgb

# Train a LightGBM model
lgb_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=7, random_state=42)
lgb_model.fit(X_train, y_train)

# Evaluate
y_pred_proba = lgb_model.predict_proba(X_test)[:, 1]
score = roc_auc_score(y_test, y_pred_proba)
print(f"LightGBM ROC-AUC Score: {score:.4f}")
